# LangGraph: Tool Use와 ReAct 패턴

이전 노트북에서 LangGraph의 기본 개념(State, Node, Edge, Graph)을 학습했습니다. 이번에는 **LLM이 도구를 사용하는 패턴**을 구현합니다.

## 개요

| 주제 | 내용 |
|------|------|
| Tool Use | LLM이 외부 도구(함수)를 호출하는 패턴 |
| bind_tools | LLM에 사용 가능한 도구 목록을 알려주기 |
| ToolNode | 도구 실행을 담당하는 미리 만들어진 노드 |
| tools_condition | 도구 호출 여부에 따라 분기하는 조건부 Edge |
| MemorySaver | 대화 기록을 유지하는 체크포인터 |
| Async 실행 | 비동기 실행으로 브라우저 도구 등 활용 |
| Playwright | 웹 브라우저를 도구로 사용하는 LangChain 커뮤니티 도구킷 |

## 학습 목표

1. LLM이 도구를 호출하는 **ReAct 패턴**의 동작 원리를 이해하기
2. `bind_tools`로 LLM에 도구를 연결하고, `ToolNode`로 실행하기
3. `tools_condition`으로 **조건부 분기**를 구현하기
4. `MemorySaver`로 대화 기록을 **영속적으로 유지**하기
5. Playwright 브라우저 도구킷으로 **웹 탐색 에이전트** 만들기

---

## 이전 노트북과의 비교

| | 04-1 (Basic) | 04-2 (Tool Use) |
|---|---|---|
| **노드** | 단일 노드 (chatbot) | chatbot + tools (2개 노드) |
| **Edge** | 일직선 (START→chatbot→END) | **순환** (chatbot ↔ tools) |
| **LLM** | 단순 응답 생성 | 도구 호출 판단 + 응답 생성 |
| **메모리** | 없음 (매 턴 독립) | MemorySaver (대화 기록 유지) |
| **실행** | 동기 (invoke) | 동기 + **비동기** (ainvoke) |

---

## 1. Tool Use와 ReAct 패턴

### 왜 도구가 필요한가?

LLM은 학습 데이터에 기반한 **텍스트 생성기**입니다. 다음과 같은 한계가 있습니다:

- 실시간 정보를 모릅니다 ("오늘 날씨는?")
- 정확한 계산을 못합니다 ("1847 × 293 = ?")
- 외부 시스템과 상호작용할 수 없습니다 ("이메일 보내줘")

**도구(Tool)**는 이 한계를 극복합니다. LLM이 "이건 내가 직접 하는 것보다 도구를 쓰는 게 낫겠다"고 **판단**하면, 도구 호출을 요청합니다.

### ReAct 패턴 (Reason + Act)

LangGraph에서 Tool Use는 **ReAct 루프**로 구현됩니다:

```
┌─────────────────────────────────────────────────────────────────────┐
│                       ReAct 루프                                   │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   사용자: "서울 날씨 알려줘"                                       │
│       │                                                             │
│       ▼                                                             │
│   ┌──────────┐                                                     │
│   │ chatbot  │  Reason: "날씨를 알려면 도구를 써야겠다"            │
│   │ (LLM)    │  → tool_call: get_weather("서울")                   │
│   └──────────┘                                                     │
│       │  도구 호출 감지 (tools_condition)                           │
│       ▼                                                             │
│   ┌──────────┐                                                     │
│   │  tools   │  Act: get_weather("서울") 실행                      │
│   │(ToolNode)│  → 결과: "맑음, 22°C"                              │
│   └──────────┘                                                     │
│       │  결과를 다시 LLM에게                                        │
│       ▼                                                             │
│   ┌──────────┐                                                     │
│   │ chatbot  │  Reason: "결과를 받았으니 사용자에게 답하자"        │
│   │ (LLM)    │  → "서울은 현재 맑고 22°C입니다"                   │
│   └──────────┘                                                     │
│       │  도구 호출 없음 → END                                      │
│       ▼                                                             │
│   사용자에게 최종 응답 전달                                         │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

핵심은 **LLM이 2번 호출**된다는 것입니다:
1. **1차**: 도구를 써야 할지 판단 → 도구 호출 요청
2. **2차**: 도구 결과를 받아서 최종 응답 생성

도구가 필요 없는 질문("안녕하세요")이면 1차에서 바로 응답하고 끝납니다.

### LangGraph의 3가지 핵심 구성요소

ReAct 루프를 구현하기 위해 LangGraph가 제공하는 3가지:

```
┌─────────────────────────────────────────────────────────────────────┐
│                                                                     │
│  1. bind_tools(tools)                                              │
│     LLM에게 "이런 도구들을 쓸 수 있어"라고 알려줌                 │
│     llm_with_tools = llm.bind_tools([get_weather, calculator])     │
│                                                                     │
│  2. ToolNode(tools)                                                │
│     LLM이 요청한 도구 호출을 실제로 실행하는 노드                  │
│     tool_node = ToolNode(tools=[get_weather, calculator])          │
│                                                                     │
│  3. tools_condition                                                │
│     LLM 응답에 도구 호출이 있는지 확인하는 조건부 Edge             │
│     → 도구 호출 있음: "tools" 노드로 이동                         │
│     → 도구 호출 없음: END로 이동                                   │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

| 구성요소 | 역할 | import 경로 |
|---------|------|-------------|
| `bind_tools` | LLM에 도구 목록 연결 | `ChatOpenAI`의 메서드 |
| `ToolNode` | 도구 실행 노드 | `langgraph.prebuilt` |
| `tools_condition` | 도구 호출 여부 분기 | `langgraph.prebuilt` |

---

## 2. 환경 설정

In [ ]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain.agents import Tool
from pydantic import BaseModel
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
import random
import os

load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print("API key found.")
else:
    print("No API key was found — .env 파일에 OPENAI_API_KEY를 설정하세요.")

---

## 3. Tool 정의하기

LangGraph에서 Tool을 정의하는 방법은 여러 가지가 있습니다. 가장 간단한 방법부터 살펴봅니다.

### 방법 1: `langchain.agents.Tool`로 감싸기

기존 Python 함수를 `Tool` 객체로 감쌉니다. **name**과 **description**이 중요합니다 — LLM은 description을 읽고 이 도구를 언제 써야 할지 판단합니다.

In [ ]:
# 간단한 도구들 정의

def get_weather(city: str) -> str:
    """도시의 현재 날씨를 반환합니다 (데모용 하드코딩)"""
    weather_data = {
        "서울": "맑음, 18°C",
        "부산": "흐림, 22°C",
        "제주": "비, 20°C",
    }
    return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")

def calculator(expression: str) -> str:
    """수학 계산식을 평가합니다"""
    try:
        # 안전한 수학 연산만 허용
        allowed = set('0123456789+-*/.() ')
        if all(c in allowed for c in expression):
            result = eval(expression)
            return str(result)
        return "허용되지 않는 문자가 포함되어 있습니다."
    except Exception as e:
        return f"계산 오류: {e}"

# Tool 객체로 감싸기
tool_weather = Tool(
    name="get_weather",
    func=get_weather,
    description="도시의 현재 날씨를 조회합니다. 입력: 도시 이름 (예: 서울)"
)

tool_calculator = Tool(
    name="calculator",
    func=calculator,
    description="수학 계산을 수행합니다. 입력: 수학 표현식 (예: 1847 * 293)"
)

tools = [tool_weather, tool_calculator]

# 도구 목록 확인
for tool in tools:
    print(f"  {tool.name}: {tool.description}")

---

## 4. bind_tools — LLM에 도구 연결하기

`bind_tools`는 LLM에게 **"이런 도구들을 사용할 수 있다"**고 알려줍니다. LLM은 도구의 name과 description을 보고, 사용자 질문에 적합한 도구가 있으면 **tool_call**을 응답에 포함합니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│  bind_tools의 동작                                                 │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  llm = ChatOpenAI(model="gpt-4o-mini")                            │
│  llm_with_tools = llm.bind_tools(tools)                           │
│                                                                     │
│  ── 도구가 필요한 질문 ──                                          │
│  "서울 날씨 알려줘"                                                │
│    → 응답에 tool_calls 포함:                                       │
│      [{name: "get_weather", args: {"city": "서울"}}]              │
│    → content는 비어있음 ("")                                       │
│                                                                     │
│  ── 도구가 불필요한 질문 ──                                        │
│  "안녕하세요"                                                      │
│    → 응답에 tool_calls 없음: []                                    │
│    → content에 직접 응답: "안녕하세요! 무엇을 도와드릴까요?"      │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

이 **tool_calls의 유무**가 이후 `tools_condition`이 분기하는 기준입니다.

In [ ]:
# bind_tools 동작 확인

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

# Case 1: 도구가 필요한 질문
response1 = llm_with_tools.invoke("서울 날씨 알려줘")
print("=== Case 1: 도구가 필요한 질문 ===")
print(f"  content: '{response1.content}'")
print(f"  tool_calls: {response1.tool_calls}")

print()

# Case 2: 도구가 불필요한 질문
response2 = llm_with_tools.invoke("안녕하세요")
print("=== Case 2: 도구가 불필요한 질문 ===")
print(f"  content: '{response2.content}'")
print(f"  tool_calls: {response2.tool_calls}")

`tool_calls`가 있으면 LLM이 "이 도구를 이 인자로 호출해달라"고 요청한 것입니다. 이 요청을 실제로 실행하는 것이 `ToolNode`입니다.

---

## 5. 그래프 구성 — ReAct 루프 구현

이제 Tool Use 그래프를 5단계로 구성합니다. 이전 노트북의 일직선 그래프와 달리, **순환(루프)**이 있는 그래프입니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│             Tool Use 그래프 구조                                   │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│         START                                                      │
│           │                                                        │
│           ▼                                                        │
│     ┌──────────┐                                                   │
│     │ chatbot  │ ◄─────────────────────┐                          │
│     │ (LLM)    │                       │                          │
│     └──────────┘                       │                          │
│      │         │                       │                          │
│      │         │                       │                          │
│  tool_calls  tool_calls               │                          │
│  없음 ✗      있음 ✓                   │                          │
│      │         │                       │                          │
│      ▼         ▼                       │                          │
│    END    ┌──────────┐                 │                          │
│           │  tools   │ ────────────────┘                          │
│           │(ToolNode)│    도구 결과를 chatbot에게 전달             │
│           └──────────┘                                             │
│                                                                     │
│   tools_condition이 분기를 결정                                    │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

**04-1과의 결정적 차이**: Edge가 `chatbot → END`로 끝나지 않고, `chatbot → tools → chatbot`으로 **순환**합니다.

In [ ]:
# ── Step 1: State 정의 ──

class State(BaseModel):
    messages: Annotated[list, add_messages]

In [ ]:
# ── Step 2 & 3: Graph Builder 생성 + Node 추가 ──

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    """LLM이 응답을 생성하거나, 도구 호출을 요청합니다."""
    return {"messages": [llm_with_tools.invoke(state.messages)]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))   # ← 미리 만들어진 노드

print("노드 추가 완료: chatbot, tools")

### ToolNode와 tools_condition 상세 설명

**`ToolNode`**는 LangGraph가 미리 만들어둔 노드입니다. 내부 동작:
1. State의 마지막 메시지에서 `tool_calls`를 확인
2. 해당하는 도구 함수를 찾아서 실행
3. 도구 실행 결과를 `ToolMessage`로 State에 추가

**`tools_condition`**은 미리 만들어둔 조건부 Edge 함수입니다. 내부 동작:
1. chatbot 노드의 응답(마지막 메시지)을 확인
2. `tool_calls`가 있으면 → `"tools"` 반환 (tools 노드로 이동)
3. `tool_calls`가 없으면 → `END` 반환 (종료)

```python
# tools_condition의 내부 로직 (단순화)
def tools_condition(state):
    last_message = state.messages[-1]
    if last_message.tool_calls:   # 도구 호출 요청이 있으면
        return "tools"            # → tools 노드로
    return END                    # → 종료
```

In [ ]:
# ── Step 4: Edge 연결 ──

# START → chatbot (항상)
graph_builder.add_edge(START, "chatbot")

# chatbot → tools 또는 END (조건부)
graph_builder.add_conditional_edges("chatbot", tools_condition, "tools")

# tools → chatbot (항상: 도구 실행 결과를 LLM에게 전달)
graph_builder.add_edge("tools", "chatbot")

print("Edge 연결 완료")

### `add_conditional_edges` 이해하기

```python
graph_builder.add_conditional_edges("chatbot", tools_condition, "tools")
#                                    ───────   ───────────────  ───────
#                                    출발 노드   조건 함수       도착 노드
```

- `tools_condition`이 `"tools"`를 반환하면 → `"tools"` 노드로 이동
- `tools_condition`이 `END`를 반환하면 → 그래프 종료

일반 Edge(`add_edge`)와의 차이:

| | `add_edge` | `add_conditional_edges` |
|---|---|---|
| **분기** | 없음 (항상 같은 곳) | 조건에 따라 다른 곳 |
| **용도** | 확정된 흐름 | 동적 분기 |
| **예시** | START→chatbot | chatbot→tools **또는** END |

In [ ]:
# ── Step 5: Compile & 시각화 ──

graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

chatbot에서 tools로, 다시 chatbot으로 돌아오는 **순환 구조**가 보입니다. 이것이 ReAct 루프입니다.

### 실행해봅시다!

In [ ]:
# 도구가 필요한 질문
result = graph.invoke({"messages": [{"role": "user", "content": "서울 날씨 알려줘"}]})

print("=== 전체 메시지 흐름 ===")
for msg in result["messages"]:
    role = msg.type if hasattr(msg, 'type') else msg.get('role', '?')
    content = msg.content if hasattr(msg, 'content') else msg.get('content', '')
    tool_calls = getattr(msg, 'tool_calls', [])
    
    if tool_calls:
        print(f"  [{role}] tool_call → {tool_calls[0]['name']}({tool_calls[0]['args']})")
    else:
        print(f"  [{role}] {content[:100]}")

In [ ]:
# 도구가 불필요한 질문 — 루프 없이 바로 응답
result = graph.invoke({"messages": [{"role": "user", "content": "안녕하세요!"}]})

print("=== 전체 메시지 흐름 ===")
for msg in result["messages"]:
    role = msg.type if hasattr(msg, 'type') else msg.get('role', '?')
    content = msg.content if hasattr(msg, 'content') else msg.get('content', '')
    print(f"  [{role}] {content[:100]}")

**메시지 흐름 비교**:

```
도구 사용 시 (4개 메시지):              도구 미사용 시 (2개 메시지):
  [human]    "서울 날씨 알려줘"          [human]    "안녕하세요!"
  [ai]       tool_call → get_weather     [ai]       "안녕하세요! ..."
  [tool]     "맑음, 18°C"                           ← 바로 끝
  [ai]       "서울은 현재 맑고 18°C..."
```

---

## 6. MemorySaver — 대화 기록 유지하기

위 그래프는 매번 `invoke`할 때마다 **이전 대화를 잊어버립니다**. 대화형 챗봇이라면 이전 대화를 기억해야 합니다.

### 문제 상황

```
사용자: "서울 날씨 알려줘"       → "맑음, 18°C입니다"
사용자: "부산은?"               → "무슨 말씀이신지 모르겠습니다" ← 이전 대화를 모름!
```

### 해결: MemorySaver (체크포인터)

`MemorySaver`는 **thread_id별로 상태를 저장**합니다. 같은 thread_id로 요청하면 이전 대화가 자동으로 이어집니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│  MemorySaver 동작 원리                                             │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  thread_id = "user_123"                                            │
│                                                                     │
│  [1차 invoke]                                                      │
│    입력: messages = [user: "서울 날씨"]                            │
│    저장: messages = [user: "서울 날씨", ai: "맑음, 18°C"]        │
│                                                                     │
│  [2차 invoke]                                                      │
│    입력: messages = [user: "부산은?"]                              │
│    복원 + 합산:                                                    │
│      messages = [user: "서울 날씨",                               │
│                  ai: "맑음, 18°C",                                │
│                  user: "부산은?"]    ← 이전 대화가 보존됨!        │
│    저장: messages = [..., ai: "부산은 흐리고 22°C입니다"]         │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

`MemorySaver`는 **인메모리** 저장소입니다 (프로세스 종료 시 소멸). 영속 저장이 필요하면 `SqliteSaver` 등을 사용합니다.

In [ ]:
# MemorySaver를 추가하여 그래프 재구성

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))
graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition, "tools")
graph_builder.add_edge("tools", "chatbot")

# checkpointer 추가 — 이것만 다름!
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

print("MemorySaver가 적용된 그래프 생성 완료")
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# MemorySaver 동작 확인
# config에 thread_id를 지정하면 같은 대화가 이어집니다

config = {"configurable": {"thread_id": "demo_1"}}

# 1차 대화
result1 = graph.invoke(
    {"messages": [{"role": "user", "content": "서울 날씨 알려줘"}]},
    config=config
)
print("[1차] 응답:", result1["messages"][-1].content)

print()

# 2차 대화 — "부산은?"이라고만 해도 맥락을 이해
result2 = graph.invoke(
    {"messages": [{"role": "user", "content": "부산은?"}]},
    config=config
)
print("[2차] 응답:", result2["messages"][-1].content)

print()

# 3차 대화 — 계산 도구 사용
result3 = graph.invoke(
    {"messages": [{"role": "user", "content": "1847 * 293 은 얼마야?"}]},
    config=config
)
print("[3차] 응답:", result3["messages"][-1].content)

In [ ]:
# 대화 기록 전체 확인

print("=== thread_id='demo_1'의 전체 대화 기록 ===")
for msg in result3["messages"]:
    role = msg.type if hasattr(msg, 'type') else msg.get('role', '?')
    content = msg.content if hasattr(msg, 'content') else msg.get('content', '')
    tool_calls = getattr(msg, 'tool_calls', [])
    
    if tool_calls:
        print(f"  [{role}] tool_call → {tool_calls[0]['name']}({tool_calls[0]['args']})")
    elif content:
        print(f"  [{role}] {content[:80]}")

### thread_id의 역할

- 같은 `thread_id` → 같은 대화 (이전 메시지 유지)
- 다른 `thread_id` → 별도의 대화 (서로 독립)

```
thread_id="user_A"  →  [서울 날씨 → 맑음 → 부산은? → 흐림]
thread_id="user_B"  →  [안녕 → 반가워요]                      ← A의 대화를 모름
```

실제 서비스에서는 사용자 ID나 세션 ID를 thread_id로 사용합니다.

---

## 7. Gradio UI로 Tool Use 챗봇 실행

In [ ]:
# Gradio UI — MemorySaver + Tool Use 챗봇

config = {"configurable": {"thread_id": "gradio_session"}}

def chat(user_input: str, history):
    result = graph.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config
    )
    return result["messages"][-1].content

gr.ChatInterface(
    chat,
    type="messages",
    title="LangGraph Tool Use 챗봇",
    description="날씨 조회와 계산이 가능한 챗봇입니다. 예: '서울 날씨', '1847 * 293'",
).launch()

---

## 8. 비동기(Async) 실행

지금까지는 **동기** 방식(`invoke`, `run`)을 사용했습니다. 하지만 웹 브라우저 조작처럼 **I/O 대기가 긴 작업**에는 **비동기** 방식이 효율적입니다.

| | 동기 (Sync) | 비동기 (Async) |
|---|---|---|
| **도구 실행** | `tool.run(inputs)` | `await tool.arun(inputs)` |
| **그래프 실행** | `graph.invoke(state)` | `await graph.ainvoke(state)` |
| **장점** | 단순함 | I/O 대기 중 다른 작업 가능 |
| **적합한 경우** | 간단한 도구 | 브라우저, 네트워크, DB 등 |

Jupyter 노트북에서 `async/await`를 사용하려면 `nest_asyncio`가 필요합니다 — Python은 기본적으로 하나의 이벤트 루프만 허용하기 때문입니다.

In [ ]:
# nest_asyncio — Jupyter에서 async를 사용하기 위한 패치
import nest_asyncio
nest_asyncio.apply()

# 비동기 실행 예시
config = {"configurable": {"thread_id": "async_demo"}}

result = await graph.ainvoke(
    {"messages": [{"role": "user", "content": "1234 + 5678 계산해줘"}]},
    config=config
)
print("비동기 실행 결과:", result["messages"][-1].content)

---

## 9. Playwright 브라우저 도구 (LangChain 커뮤니티)

LangChain 커뮤니티에는 수많은 미리 만들어진 도구가 있습니다. 그중 **Playwright**는 실제 웹 브라우저를 프로그래밍으로 제어하는 도구입니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│  Playwright 브라우저 도구킷                                        │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  navigate_browser   →  URL로 이동                                  │
│  extract_text       →  페이지 텍스트 추출                          │
│  click_element      →  버튼/링크 클릭                              │
│  fill_text          →  입력 필드에 텍스트 입력                     │
│  get_elements       →  HTML 요소 검색                              │
│  current_page       →  현재 URL 확인                               │
│                                                                     │
│  → LLM이 이 도구들을 조합하여 웹사이트를 탐색합니다               │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### 사전 준비

Playwright를 사용하려면 Node.js와 브라우저가 설치되어 있어야 합니다:

```bash
pip install playwright langchain-community
playwright install
```

> **참고**: Windows에서 Jupyter 노트북 실행 시 `NotImplementedError`가 발생할 수 있습니다. 이는 Windows의 이벤트 루프 정책 문제로, Python 스크립트로 실행하면 해결됩니다.

In [ ]:
# Playwright 브라우저 도구 설정
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser

# headless=False로 하면 실제 브라우저 창이 열립니다
async_browser = create_async_playwright_browser(headless=False)
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
browser_tools = toolkit.get_tools()

# 어떤 도구들이 있는지 확인
print("=== Playwright 브라우저 도구 목록 ===")
for tool in browser_tools:
    print(f"  {tool.name}: {tool.description[:60]}...")

In [ ]:
# 브라우저 도구 직접 사용해보기
import textwrap

tool_dict = {tool.name: tool for tool in browser_tools}
navigate_tool = tool_dict["navigate_browser"]
extract_text_tool = tool_dict["extract_text"]

# 웹사이트 방문
await navigate_tool.arun({"url": "https://www.example.com"})

# 텍스트 추출
text = await extract_text_tool.arun({})
print(textwrap.fill(text, width=80))

---

## 10. 전체 통합 — 브라우저 + 도구 챗봇

Playwright 브라우저 도구와 기존 도구(날씨, 계산)를 모두 합쳐서 **만능 챗봇**을 만듭니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│                통합 에이전트 아키텍처                               │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  사용자: "CNN 뉴스 헤드라인 알려줘"                                │
│       │                                                             │
│       ▼                                                             │
│  ┌──────────┐   도구 선택:                                         │
│  │ chatbot  │   - navigate_browser("https://cnn.com")              │
│  │ (LLM)    │   - extract_text()                                   │
│  └──────────┘                                                      │
│       │                                                             │
│       ▼                                                             │
│  ┌──────────┐   Playwright로 실제 브라우저 실행                    │
│  │  tools   │   → CNN 웹사이트 방문 + 텍스트 추출                  │
│  └──────────┘                                                      │
│       │                                                             │
│       ▼                                                             │
│  ┌──────────┐                                                      │
│  │ chatbot  │   추출된 텍스트를 요약하여 응답                      │
│  │ (LLM)    │   → "현재 CNN 헤드라인: 1. ... 2. ... 3. ..."       │
│  └──────────┘                                                      │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# 모든 도구 통합
all_tools = browser_tools + tools  # Playwright 도구 + 날씨/계산 도구

print(f"총 {len(all_tools)}개 도구:")
for t in all_tools:
    print(f"  - {t.name}")

In [ ]:
# 통합 그래프 구성

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_all_tools = llm.bind_tools(all_tools)

def chatbot_with_all_tools(state: State):
    return {"messages": [llm_with_all_tools.invoke(state.messages)]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot_with_all_tools)
graph_builder.add_node("tools", ToolNode(tools=all_tools))
graph_builder.add_conditional_edges("chatbot", tools_condition, "tools")
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# 비동기 Gradio UI — 브라우저 + 도구 통합 챗봇

config = {"configurable": {"thread_id": "full_agent"}}

async def chat(user_input: str, history):
    result = await graph.ainvoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config
    )
    return result["messages"][-1].content

gr.ChatInterface(
    chat,
    type="messages",
    title="LangGraph 통합 에이전트",
    description="날씨 조회, 계산, 웹 브라우저 탐색이 가능합니다. 예: 'CNN 뉴스 헤드라인 요약해줘'"
).launch()

---

## 정리

### Tool Use 그래프 구성 패턴

```
┌─────────────────────────────────────────────────────────────────────┐
│                 Tool Use 핵심 요약                                  │
├──────────────────────────────────┬──────────────────────────────────┤
│     구성요소                     │     코드                         │
├──────────────────────────────────┼──────────────────────────────────┤
│  도구를 LLM에 연결               │  llm.bind_tools(tools)          │
│  도구 실행 노드                  │  ToolNode(tools=tools)          │
│  조건부 분기                     │  tools_condition                │
│  대화 기억                       │  MemorySaver()                  │
│  비동기 실행                     │  await graph.ainvoke(state)     │
└──────────────────────────────────┴──────────────────────────────────┘
```

### 핵심 포인트

- **ReAct 루프**: chatbot → tools → chatbot 순환으로 도구 사용
- **tools_condition**: LLM 응답에 `tool_calls`가 있으면 tools로, 없으면 END로 분기
- **MemorySaver**: `thread_id`별로 대화 기록을 유지하는 체크포인터
- **LangChain 커뮤니티**: Playwright 등 수많은 미리 만들어진 도구킷 활용 가능

### 04-1과의 차이

| | 04-1 (Basic) | 04-2 (Tool Use) |
|---|---|---|
| 그래프 형태 | 일직선 | **순환 (ReAct 루프)** |
| Edge | 일반 Edge만 | 일반 + **조건부 Edge** |
| 메모리 | 없음 | **MemorySaver** |
| LLM 역할 | 단순 응답 | **도구 호출 판단 + 응답** |

### 다음 단계

- **Human-in-the-loop**: 도구 실행 전 사용자 승인 요청
- **Multi-agent**: 여러 에이전트가 협업하는 그래프
- **Streaming**: 응답을 실시간으로 스트리밍